#Quality check proof - Retail Orders Dataset


This notebook validates the cleaned dataset after the data cleaning process.


Main validation checks:
1. Missing values check
2. Duplicate row check
3. Duplicate orderID check
4. Date format check
5. Numeric value check
6. TotalPrice validation
7. Final quality summary

In [5]:
import pandas as pd
from pathlib import Path

In [6]:
cleaned_data_path = Path("../data/processed/orders_cleaned.csv")
df = pd.read_csv(cleaned_data_path)
print("Cleaned dataset loaded successfully")
print("Shape:", df.shape)

df.head()

Cleaned dataset loaded successfully
Shape: (1200, 14)


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [7]:
missing_values = df.isnull().sum()

missing_report = pd.DataFrame({
    "Column": missing_values.index,
    "Missing Values": missing_values.values
})
missing_report

,Column,Missing Values
0,OrderID,0
1,Date,0
2,CustomerID,0
3,Product,0
4,Quantity,0
5,UnitPrice,0
6,ShippingAddress,0
7,PaymentMethod,0
8,OrderStatus,0
9,TrackingNumber,0


In [8]:
total_missing_values = df.isnull().sum().sum()
print("Total missing values:", total_missing_values)

Total missing values: 0


In [9]:
duplicate_rows = df.duplicated().sum()
print("duplicate rows:", duplicate_rows)

duplicate rows: 0


In [10]:
duplicate_order_ids = df["OrderID"].duplicated().sum()
print("Duplicate OrderID values:", duplicate_order_ids)

Duplicate OrderID values: 0


In [12]:
converted_dates = pd.to_datetime(df["Date"], errors = "coerce") #convert date column to datetime
invalid_dates = converted_dates.isnull().sum() #count invalid dates
print("Invalid dates:", invalid_dates)

Invalid dates: 0


In [14]:
date_format_check = df["Date"].astype(str).str.match(r"^\d{4}-\d{2}-\d{2}$")
incorrect_date_format_count = (~date_format_check).sum()
print("Incorrect YYYY-MM-DD date formates:", incorrect_date_format_count)

Incorrect YYYY-MM-DD date formates: 0


In [15]:
numeric_columns = ["Quantity", "UnitPrice", "ItemsInCart", "TotalPrice"]

for col in numeric_columns:
    print(f"\nColumn: {col}")
    print("Missing values:", df[col].isnull().sum())
    print("Minimum value:", df[col].min())
    print("Maximum value:", df[col].max())


Column: Quantity
Missing values: 0
Minimum value: 1
Maximum value: 5

Column: UnitPrice
Missing values: 0
Minimum value: 11.39
Maximum value: 699.93

Column: ItemsInCart
Missing values: 0
Minimum value: 1
Maximum value: 10

Column: TotalPrice
Missing values: 0
Minimum value: 11.39
Maximum value: 3456.4


In [16]:
negative_quantity = (df["Quantity"] < 0).sum()
negative_unit_price = (df["UnitPrice"] < 0).sum()
negative_items_in_cart = (df["ItemsInCart"] < 0).sum()
negative_total_price = (df["TotalPrice"] < 0).sum()

print("Negative Quantity values:", negative_quantity)
print("Negative UnitPrice values:", negative_unit_price)
print("Negative ItemsInCart values:", negative_items_in_cart)
print("Negative TotalPrice values:", negative_total_price)

Negative Quantity values: 0
Negative UnitPrice values: 0
Negative ItemsInCart values: 0
Negative TotalPrice values: 0


In [17]:
expected_total_price = (df["Quantity"] * df["UnitPrice"]).round(2)

incorrect_total_price = (df["TotalPrice"].round(2) != expected_total_price).sum()

print("Incorrect TotalPrice rows:", incorrect_total_price)

Incorrect TotalPrice rows: 0


In [18]:
quality_summary = {
    "Total Rows": df.shape[0],
    "Total Columns": df.shape[1],
    "Total Missing Values": total_missing_values,
    "Duplicate Rows": duplicate_rows,
    "Duplicate OrderID Values": duplicate_order_ids,
    "Invalid Dates": invalid_dates,
    "Incorrect Date Formats": incorrect_date_format_count,
    "Negative Quantity Values": negative_quantity,
    "Negative UnitPrice Values": negative_unit_price,
    "Negative ItemsInCart Values": negative_items_in_cart,
    "Negative TotalPrice Values": negative_total_price,
    "Incorrect TotalPrice Rows": incorrect_total_price
}

quality_summary_df = pd.DataFrame(
    list(quality_summary.items()),
    columns=["Quality Check", "Result"]
)

quality_summary_df

,Quality Check,Result
0,Total Rows,1200
1,Total Columns,14
2,Total Missing Values,0
3,Duplicate Rows,0
4,Duplicate OrderID Values,0
5,Invalid Dates,0
6,Incorrect Date Formats,0
7,Negative Quantity Values,0
8,Negative UnitPrice Values,0
9,Negative ItemsInCart Values,0


In [19]:
reports_folder = Path("../docs")
reports_folder.mkdir(parents=True, exist_ok=True)

quality_report_path = reports_folder / "quality_check_summary.csv"

quality_summary_df.to_csv(quality_report_path, index=False)

print("Quality check summary saved to:", quality_report_path)

Quality check summary saved to: ..\docs\quality_check_summary.csv
